In [0]:
display(spark.sql(
    """
    SELECT 
        MIN(transaction_date) AS min_date,
        MAX(transaction_date) AS max_date
    FROM fraud_detection.bronze.transactions_raw
    """
))

In [0]:
import requests

start_date = "2024-01-01"
end_date = "2024-04-03"

resp = requests.get(f"https://api.frankfurter.dev/v2/rates?base=USD&from={start_date}&to={end_date}")
resp.raise_for_status()
data = resp.json()

# v2 returns a list of records, each with date, base, quote, and rate
import json
print(f"Type: {type(data)}")
print(f"Total records: {len(data)}")
print(json.dumps(data[:5], indent=2))

In [0]:
from pyspark.sql import functions as F

fx_rows = [{"rate_date": r["date"], "currency": r["quote"], "rate_to_usd": float(r["rate"])} for r in data]
fx_df = spark.createDataFrame(fx_rows)

# USD itself is omitted as its own base currency - add it explicitly at rate 1.0
distinct_dates = fx_df.select("rate_date").distinct()
usd_rows = distinct_dates.withColumn("currency", F.lit("USD")).withColumn("rate_to_usd", F.lit(1.0))

fx_df_complete = fx_df.select("rate_date", "currency", "rate_to_usd").union(usd_rows)

print(f"Total rows: {fx_df_complete.count()}")
print(f"Distinct currencies: {fx_df_complete.select('currency').distinct().count()}")

In [0]:
fx_df_complete.write.mode("overwrite").saveAsTable("fraud_detection.bronze.fx_rates_raw")
display(spark.sql("SELECT COUNT(*) FROM fraud_detection.bronze.fx_rates_raw"))